# 01 — EDA: Power Production
**Input:** `data/raw/Zeitreihen_WKA_2025_2026.xlsx`  
**Plants:** Unterpreilipp (Rudolstadt) · Burgau (Jena)  
**Resolution:** 15 min · **Timezone raw:** UTC+1 → converted to UTC here  
**Period:** Jan 2025 – May 2026 (~16 months)

**Goals**
1. Load & clean the raw Excel (fix timezone, drop duplicate, cast types)
2. Inspect coverage — gaps, zeros, shutdown periods
3. Time series overview — daily/weekly/monthly patterns
4. Distribution & outliers per plant
5. Autocorrelation — validates the seasonal-persistence baseline assumption


In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pandas.plotting import autocorrelation_plot

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 4)})


## 1 · Load & Clean

In [ ]:
RAW_PATH = "../data/raw/Zeitreihen_WKA_2025_2026.xlsx"

# Rows 0-4 are metadata; row 4 is the real header
raw = pd.read_excel(RAW_PATH, header=None)

wka = raw.iloc[6:].copy() 
wka.columns = ["Datum_von", "Datum_bis", "Unterpreilipp_kW", "Burgau_kW"]

# Parse timestamps — data is UTC+1 (CET), shift to UTC
wka["Datum_von"] = (
    (pd.to_datetime(wka["Datum_von"], dayfirst=True) - pd.Timedelta(hours=1))
    .dt.tz_localize("UTC")
)

wka["Unterpreilipp_kW"] = pd.to_numeric(wka["Unterpreilipp_kW"], errors="coerce")
wka["Burgau_kW"]        = pd.to_numeric(wka["Burgau_kW"],        errors="coerce")

# Drop duplicate timestamps (there is exactly 1)
wka = wka.drop_duplicates(subset="Datum_von")
wka = wka.set_index("Datum_von").drop(columns=["Datum_bis"])
wka = wka.sort_index()

print("Shape      :", wka.shape)
print("Range      :", wka.index.min(), "→", wka.index.max())
print("Inferred freq:", pd.infer_freq(wka.index))
print("Nulls      :", wka.isnull().sum().to_dict())
wka.head(3)


DateParseError: Unknown datetime string format, unable to parse: Datum_von

## 2 · Coverage — Gaps & Zeros

In [ ]:
# Check for missing 15-min slots
expected = pd.date_range(wka.index.min(), wka.index.max(), freq="15min", tz="UTC")
missing_slots = expected.difference(wka.index)
print(f"Expected slots : {len(expected):,}")
print(f"Actual rows    : {len(wka):,}")
print(f"Missing slots  : {len(missing_slots)}")
if len(missing_slots):
    print("First missing  :", missing_slots[:3].tolist())


In [ ]:
# Zero / shutdown analysis
for col in ["Unterpreilipp_kW", "Burgau_kW"]:
    zeros = (wka[col] == 0).sum()
    pct   = zeros / len(wka) * 100
    print(f"{col:25s}  zeros: {zeros:5,}  ({pct:.1f}%)")


In [ ]:
# Visualise shutdown periods (zeros shown as red spans)
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

for ax, col, color in zip(axes, wka.columns, ["steelblue", "darkorange"]):
    daily = wka[col].resample("D").mean()
    ax.plot(daily.index, daily, lw=0.8, color=color, label=col)
    # Shade zero-production days
    zero_days = daily[daily == 0].index
    for d in zero_days:
        ax.axvspan(d, d + pd.Timedelta("1D"), alpha=0.25, color="red", lw=0)
    ax.set_ylabel("kW (daily mean)")
    ax.set_title(col)
    ax.legend(fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.suptitle("Daily mean production — red = zero-production days", y=1.01)
plt.tight_layout()
plt.show()


## 3 · Time-Series Overview

In [ ]:
# Resample to hourly for a cleaner visual
hourly = wka.resample("1h").mean()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(hourly.index, hourly["Unterpreilipp_kW"], lw=0.6, label="Unterpreilipp", alpha=0.8)
ax.plot(hourly.index, hourly["Burgau_kW"],        lw=0.6, label="Burgau",        alpha=0.8)
ax.set_ylabel("kW (hourly mean)")
ax.set_title("Hourly mean production — both plants")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()


In [ ]:
# Monthly box plots — seasonality check
wka_non_zero = wka[(wka["Unterpreilipp_kW"] > 0) & (wka["Burgau_kW"] > 0)].copy()
wka_non_zero["month"] = wka_non_zero.index.month

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
for ax, col in zip(axes, wka.columns):
    data_by_month = [wka_non_zero[wka_non_zero["month"] == m][col].values
                     for m in range(1, 13)]
    ax.boxplot(data_by_month, labels=range(1, 13), flierprops=dict(markersize=2))
    ax.set_xlabel("Month")
    ax.set_ylabel("kW")
    ax.set_title(f"{col} — monthly distribution (non-zero only)")
plt.tight_layout()
plt.show()


In [ ]:
# Average intra-day profile (hour of day)
wka_non_zero["hour"] = wka_non_zero.index.hour

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ["Unterpreilipp_kW", "Burgau_kW"]):
    profile = wka_non_zero.groupby("hour")[col].mean()
    ax.plot(profile.index, profile.values, marker="o", ms=4)
    ax.set_xlabel("Hour of day (UTC)")
    ax.set_ylabel("kW (mean)")
    ax.set_title(f"{col} — average intra-day profile")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 4 · Distribution & Outliers

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, color in zip(axes, wka.columns, ["steelblue", "darkorange"]):
    data = wka[col].dropna()
    ax.hist(data[data > 0], bins=80, color=color, alpha=0.75, edgecolor="none")
    ax.axvline(data[data > 0].mean(),   color="black", lw=1.2, linestyle="--", label="mean")
    ax.axvline(data[data > 0].median(), color="red",   lw=1.2, linestyle=":",  label="median")
    ax.set_xlabel("kW")
    ax.set_title(f"{col} — distribution (non-zero)")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print(wka[wka > 0].describe().round(1))


## 5 · Autocorrelation — Validates Persistence Baseline

In [ ]:
# ACF up to lag 200 (≈ 50 h) on non-zero Unterpreilipp (daily pattern → spikes at 96, 192)
from pandas.plotting import autocorrelation_plot

series = wka["Unterpreilipp_kW"].dropna()
series = series[series > 0]

fig, ax = plt.subplots(figsize=(14, 4))
autocorrelation_plot(series.iloc[:5000], ax=ax)
ax.set_xlim(0, 200)
ax.axvline(96,  color="red",   lw=1, linestyle="--", label="lag 96 (1 day)")
ax.axvline(192, color="orange", lw=1, linestyle="--", label="lag 192 (2 days)")
ax.set_title("Autocorrelation — Unterpreilipp (first 5 000 observations)")
ax.legend()
plt.tight_layout()
plt.show()
print("ACF at lag 96 (same time yesterday):", series.autocorr(lag=96).round(3))


## Summary

| Finding | Impact on modelling |
|---|---|
| UTC+1 raw timestamps | Convert to UTC at load time — done above |
| 1 duplicate row | Dropped — no effect |
| Unterpreilipp zeros ~1.9% | Minor; likely short maintenance |
| **Burgau zeros ~16.5%** | Major — treat as `plant_offline` flag; train on non-zero periods |
| Strong lag-96 autocorrelation | Validates seasonal-persistence baseline |
| Monthly seasonality visible | LightGBM `month` feature will carry signal |
